In [3]:
import tensorflow as tf
import keras
import pandas as pd
import numpy as np
import cv2
import sklearn

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("Scikit-learn:", sklearn.__version__)

TensorFlow: 2.21.0
Keras: 3.15.0
Pandas: 3.0.5
NumPy: 2.4.6
OpenCV: 5.0.0
Scikit-learn: 1.9.0


In [2]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
print(sys.executable)

/Users/aximsoft/tfenv/bin/python


In [4]:
# load data
data = pd.read_csv("dataset/HAM10000_metadata.csv")

In [5]:
import os
import glob

image_paths = {}
folders = ["dataset/HAM10000_images_part_1", "dataset/HAM10000_images_part_2"]
for folder in folders:
    for img_path in glob.glob(os.path.join(folder, "*.jpg")):
        image_id = os.path.splitext(os.path.basename(img_path))[0]
        image_paths[image_id] = img_path

print("Total images:", len(image_paths))

Total images: 10015


In [6]:
data["image_path"] = data["image_id"].map(image_paths)
print(data["image_path"].isnull().sum())

0


In [7]:
data.columns

Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'image_path'],
      dtype='str')

In [8]:
# tarin validation test-split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    stratify=data["dx"],
    random_state=42
)

In [9]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dx"],
    random_state=42
)

In [10]:
print("Training Dataset :", train_df.shape)
print("Validation Dataset :", val_df.shape)
print("Testing Dataset :", test_df.shape)

Training Dataset : (7010, 8)
Validation Dataset : (1502, 8)
Testing Dataset : (1503, 8)


In [11]:
# handli class imbalance
from sklearn.utils.class_weight import compute_class_weight


classes = np.unique(train_df["dx"])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["dx"]
)

class_weights = dict(enumerate(weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [12]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [13]:
print(data["image_path"].iloc[0])
print(type(data["image_path"].iloc[0]))

dataset/HAM10000_images_part_1/ISIC_0027419.jpg
<class 'str'>


In [14]:
data["image_path"].head()

0    dataset/HAM10000_images_part_1/ISIC_0027419.jpg
1    dataset/HAM10000_images_part_1/ISIC_0025030.jpg
2    dataset/HAM10000_images_part_1/ISIC_0026769.jpg
3    dataset/HAM10000_images_part_1/ISIC_0025661.jpg
4    dataset/HAM10000_images_part_2/ISIC_0031633.jpg
Name: image_path, dtype: str

In [15]:
print(data["image_path"].iloc[0])

print(type(data["image_path"].iloc[0]))

data["image_path"].head()

import os
print(os.path.exists(str(data["image_path"].iloc[0])))

dataset/HAM10000_images_part_1/ISIC_0027419.jpg
<class 'str'>
True


In [16]:
# resize image
IMAGE_SIZE = (224,224)
image = cv2.imread(data["image_path"].iloc[0])
resized_image = cv2.resize(image, IMAGE_SIZE)
print(resized_image.shape)

(224, 224, 3)


In [17]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,
    height_shift_range=0.1,

    horizontal_flip=True,
    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]
)

# Validation Generator
val_datagen = ImageDataGenerator(
    rescale=1./255
)
# Test Generator
test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [18]:
# create train  pipeline
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=True
)
# create validatation pipeline
val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

# create test pipeline
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


 ## Depp CNN

In [4]:
deep_results = []
deep_results.append({
    "Model": "deep CNN",
    "Train Accuracy": 47.97,
    "Validation Accuracy": 48.74,
    "Test Accuracy": 46.64,
    "Train Loss": 1.3525,
    "Validation Loss": 1.2767,
    "Test Loss": 1.3052
})

In [5]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052


## early stoping

In [19]:
deep_cnn = Sequential()

# Block 1
deep_cnn.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 2
deep_cnn.add(Conv2D(64, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 3
deep_cnn.add(Conv2D(128, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 4
deep_cnn.add(Conv2D(256, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Fully Connected Layer
deep_cnn.add(Flatten())
deep_cnn.add(Dense(256, activation="relu"))

# Output Layer (7 diseases)
deep_cnn.add(Dense(7, activation="softmax"))


In [85]:
# compile the model
deep_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [87]:
history_basic_es = deep_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 600ms/step - accuracy: 0.4800 - loss: 1.3132 - val_accuracy: 0.5193 - val_loss: 1.2539
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 609ms/step - accuracy: 0.5071 - loss: 1.2837 - val_accuracy: 0.5399 - val_loss: 1.2584
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 131s 595ms/step - accuracy: 0.5073 - loss: 1.2299 - val_accuracy: 0.5759 - val_loss: 1.0993
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 131s 595ms/step - accuracy: 0.5086 - loss: 1.2156 - val_accuracy: 0.5726 - val_loss: 1.0702
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 131s 594ms/step - accuracy: 0.5432 - loss: 1.1879 - val_accuracy: 0.5067 - val_loss: 1.3271
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 607ms/step - accuracy: 0.5280 - loss: 1.2032 - val_accuracy: 0.5726 - val_loss: 1.0450
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 608ms/step - accuracy: 0.5128 - loss: 1.2262 - val_accuracy: 0.5020 - val_loss: 1.2785
Epoch 8/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 131s 595ms/step - accuracy: 0.5454 -

In [88]:
train_loss, train_acc = deep_cnn.evaluate(train_generator)
val_loss, val_acc = deep_cnn.evaluate(val_generator)
test_loss, test_acc = deep_cnn.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 55s 249ms/step - accuracy: 0.5916 - loss: 0.9645
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 162ms/step - accuracy: 0.6039 - loss: 0.9616
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 163ms/step - accuracy: 0.5842 - loss: 0.9742


In [96]:
deep_cnn.save("models2/deep_cnn_earlystopping.keras")
print("saved deep_es")

saved deep_es


In [6]:
deep_results.append({
    "Model": "deep CNN  Early Stopping",
    "Train Accuracy": 59.16,
    "Validation Accuracy": 60.39,
    "Test Accuracy": 58.42,
    "Train Loss": 0.9645,
    "Validation Loss": 0.9616,
    "Test Loss": 0.9742
})

In [7]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,deep CNN Early Stopping,59.16,60.39,58.42,0.9645,0.9616,0.9742


## Learning rate

In [219]:
# compile the model
deep_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [221]:
deep_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [222]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

In [223]:
history_deep_lr = deep_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights,
    callbacks=[lr_scheduler]
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 143s 645ms/step - accuracy: 0.4964 - loss: 1.2908 - val_accuracy: 0.5000 - val_loss: 1.2899 - learning_rate: 1.0000e-04
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 139s 632ms/step - accuracy: 0.5137 - loss: 1.2529 - val_accuracy: 0.5666 - val_loss: 1.0618 - learning_rate: 1.0000e-04
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 143s 649ms/step - accuracy: 0.5087 - loss: 1.2388 - val_accuracy: 0.5626 - val_loss: 1.0467 - learning_rate: 1.0000e-04
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 142s 646ms/step - accuracy: 0.5177 - loss: 1.2104 - val_accuracy: 0.5613 - val_loss: 1.0974 - learning_rate: 1.0000e-04
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 140s 633ms/step - accuracy: 0.5144 - loss: 1.2500 - val_accuracy: 0.5826 - val_loss: 1.0594 - learning_rate: 1.0000e-04
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 144s 654ms/step - accuracy: 0.5268 - loss: 1.1995 - val_accuracy: 0.5646 - val_loss: 1.0301 - learning_rate: 1.0000e-04
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 

In [224]:
train_loss, train_acc = deep_cnn.evaluate(train_generator)
val_loss, val_acc = deep_cnn.evaluate(val_generator)
test_loss, test_acc = deep_cnn.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 55s 247ms/step - accuracy: 0.6076 - loss: 0.9297
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 163ms/step - accuracy: 0.6059 - loss: 0.9427
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.6008 - loss: 0.9525


In [225]:
deep_cnn.save("models2/deep_cnn_learningrate.keras")
print("saved deep_es")

saved deep_es


In [8]:
deep_results.append({
    "Model": "deep CNN Learning Rate",
    "Train Accuracy": 60.76,
    "Validation Accuracy": 60.59,
    "Test Accuracy": 60.08,
    "Train Loss": 0.9297,
    "Validation Loss": 0.9427,
    "Test Loss": 0.9525
})

In [9]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,deep CNN Early Stopping,59.16,60.39,58.42,0.9645,0.9616,0.9742
2,deep CNN Learning Rate,60.76,60.59,60.08,0.9297,0.9427,0.9525


## Optimizer Comparison

### 1.SGD

In [238]:
deep_cnn = Sequential()

# Block 1
deep_cnn.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 2
deep_cnn.add(Conv2D(64, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 3
deep_cnn.add(Conv2D(128, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 4
deep_cnn.add(Conv2D(256, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Fully Connected Layer
deep_cnn.add(Flatten())
deep_cnn.add(Dense(256, activation="relu"))

# Output Layer (7 diseases)
deep_cnn.add(Dense(7, activation="softmax"))

In [239]:
from tensorflow.keras.optimizers import SGD
 
deep_cnn.compile(
    optimizer=SGD(learning_rate=0.001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [240]:
history_deep_sgd = deep_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 131s 592ms/step - accuracy: 0.2706 - loss: 1.7364 - val_accuracy: 0.5073 - val_loss: 1.3880
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 128s 580ms/step - accuracy: 0.1090 - loss: 1.9537 - val_accuracy: 0.0340 - val_loss: 1.9257
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 127s 576ms/step - accuracy: 0.2106 - loss: 1.9360 - val_accuracy: 0.1112 - val_loss: 1.8491
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 127s 575ms/step - accuracy: 0.1656 - loss: 1.8513 - val_accuracy: 0.2304 - val_loss: 1.7235
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 128s 583ms/step - accuracy: 0.2445 - loss: 1.6442 - val_accuracy: 0.1345 - val_loss: 1.7167
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 130s 589ms/step - accuracy: 0.3364 - loss: 1.5425 - val_accuracy: 0.4061 - val_loss: 1.5926
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 129s 587ms/step - accuracy: 0.4280 - loss: 1.5145 - val_accuracy: 0.4854 - val_loss: 1.3205
Epoch 8/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 128s 581ms/step - accuracy: 0.3837 - loss: 1

In [119]:
train_loss, train_acc = deep_cnn.evaluate(train_generator)
val_loss, val_acc = deep_cnn.evaluate(val_generator)
test_loss, test_acc = deep_cnn.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 55s 248ms/step - accuracy: 0.4822 - loss: 1.2879
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.5113 - loss: 1.2163
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.4977 - loss: 1.2412


In [241]:
deep_cnn.save("models2/deep_cnn_sgd.keras")
print("saved sgd")

saved sgd


In [10]:
deep_results.append({
    "Model": "deep CNN Optimizer(SGD)",
    "Train Accuracy": 48.22,
    "Validation Accuracy": 51.13,
    "Test Accuracy": 49.77,
    "Train Loss": 1.2879,
    "Validation Loss": 1.2163,
    "Test Loss": 1.2412
})

In [11]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,deep CNN Early Stopping,59.16,60.39,58.42,0.9645,0.9616,0.9742
2,deep CNN Learning Rate,60.76,60.59,60.08,0.9297,0.9427,0.9525
3,deep CNN Optimizer(SGD),48.22,51.13,49.77,1.2879,1.2163,1.2412


### 2.RMSprop

In [23]:
deep_cnn = Sequential()

# Block 1
deep_cnn.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 2
deep_cnn.add(Conv2D(64, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 3
deep_cnn.add(Conv2D(128, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 4
deep_cnn.add(Conv2D(256, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Fully Connected Layer
deep_cnn.add(Flatten())
deep_cnn.add(Dense(256, activation="relu"))

# Output Layer (7 diseases)
deep_cnn.add(Dense(7, activation="softmax"))

In [24]:
from tensorflow.keras.optimizers import RMSprop
 
deep_cnn.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [123]:
history_deep_rmsprop = deep_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 605ms/step - accuracy: 0.3096 - loss: 1.9938 - val_accuracy: 0.4527 - val_loss: 1.3721
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 607ms/step - accuracy: 0.3837 - loss: 1.6836 - val_accuracy: 0.2663 - val_loss: 1.8665
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 139s 630ms/step - accuracy: 0.4243 - loss: 1.6068 - val_accuracy: 0.4847 - val_loss: 1.2547
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 141s 641ms/step - accuracy: 0.4342 - loss: 1.5561 - val_accuracy: 0.3362 - val_loss: 1.5624
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 604ms/step - accuracy: 0.4559 - loss: 1.5385 - val_accuracy: 0.5659 - val_loss: 1.1015
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 131s 594ms/step - accuracy: 0.4698 - loss: 1.5578 - val_accuracy: 0.5260 - val_loss: 1.1804
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 129s 583ms/step - accuracy: 0.4591 - loss: 1.5096 - val_accuracy: 0.5353 - val_loss: 1.1507
Epoch 8/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 130s 590ms/step - accuracy: 0.4907 - loss: 1

In [127]:
train_loss, train_acc = deep_cnn.evaluate(train_generator)
val_loss, val_acc = deep_cnn.evaluate(val_generator)
test_loss, test_acc = deep_cnn.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 57s 260ms/step - accuracy: 0.5157 - loss: 1.2501
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 171ms/step - accuracy: 0.5200 - loss: 1.2267
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 163ms/step - accuracy: 0.4957 - loss: 1.2492


In [130]:
deep_cnn.save("models2/deep_cnn_optimization(rmsprop).keras")
print("saved deep_rmsprop")

saved deep_rmsprop


In [12]:
deep_results.append({
    "Model": "deep CNN Optimizer(RMSprop)",
    "Train Accuracy": 51.57,
    "Validation Accuracy": 52.00,
    "Test Accuracy": 49.57,
    "Train Loss": 1.2501,
    "Validation Loss": 1.2267,
    "Test Loss": 1.2492
})

In [13]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,deep CNN Early Stopping,59.16,60.39,58.42,0.9645,0.9616,0.9742
2,deep CNN Learning Rate,60.76,60.59,60.08,0.9297,0.9427,0.9525
3,deep CNN Optimizer(SGD),48.22,51.13,49.77,1.2879,1.2163,1.2412
4,deep CNN Optimizer(RMSprop),51.57,52.00,49.57,1.2501,1.2267,1.2492


## Batch Size Comparison

### Batchsize(16)

In [133]:
deep_cnn = Sequential()

# Block 1
deep_cnn.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 2
deep_cnn.add(Conv2D(64, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 3
deep_cnn.add(Conv2D(128, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 4
deep_cnn.add(Conv2D(256, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Fully Connected Layer
deep_cnn.add(Flatten())
deep_cnn.add(Dense(256, activation="relu"))

# Output Layer (7 diseases)
deep_cnn.add(Dense(7, activation="softmax"))

In [134]:
deep_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [94]:
train_generator_16 = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)
val_generator_16 = test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)
test_generator_16 = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [136]:
history_deep_batch16 = deep_cnn.fit(
    train_generator_16,
    validation_data=val_generator_16,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 146s 331ms/step - accuracy: 0.4767 - loss: 1.3648 - val_accuracy: 0.4774 - val_loss: 1.2870
Epoch 2/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 138s 313ms/step - accuracy: 0.4890 - loss: 1.3019 - val_accuracy: 0.4980 - val_loss: 1.2803
Epoch 3/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 136s 309ms/step - accuracy: 0.5061 - loss: 1.2848 - val_accuracy: 0.5093 - val_loss: 1.1978
Epoch 4/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 136s 310ms/step - accuracy: 0.5100 - loss: 1.2369 - val_accuracy: 0.5320 - val_loss: 1.0943
Epoch 5/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 138s 313ms/step - accuracy: 0.5128 - loss: 1.2491 - val_accuracy: 0.5679 - val_loss: 1.1029
Epoch 6/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 135s 308ms/step - accuracy: 0.5298 - loss: 1.1995 - val_accuracy: 0.5905 - val_loss: 1.0442
Epoch 7/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 137s 312ms/step - accuracy: 0.5362 - loss: 1.1545 - val_accuracy: 0.5526 - val_loss: 1.0860
Epoch 8/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 139s 317ms/step - accuracy: 0.5201 - loss: 1

In [138]:
rain_loss, train_acc = deep_cnn.evaluate(train_generator_16)
val_loss, val_acc = deep_cnn.evaluate(val_generator_16)
test_loss, test_acc = deep_cnn.evaluate(test_generator_16)

439/439 ━━━━━━━━━━━━━━━━━━━━ 56s 128ms/step - accuracy: 0.4318 - loss: 1.3534
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 84ms/step - accuracy: 0.4554 - loss: 1.3208
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 83ms/step - accuracy: 0.4504 - loss: 1.3397


In [139]:
deep_cnn.save("models2/deep_cnn_batch(16).keras")
print("saved batch(16)")

saved batch(16)


In [14]:
deep_results.append({
    "Model": "deep CNN Batchsize(16))",
    "Train Accuracy": 48.22,
    "Validation Accuracy": 51.13,
    "Test Accuracy": 49.77,
    "Train Loss": 1.2879,
    "Validation Loss": 1.2163,
    "Test Loss": 1.2412
})

In [15]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,deep CNN Early Stopping,59.16,60.39,58.42,0.9645,0.9616,0.9742
2,deep CNN Learning Rate,60.76,60.59,60.08,0.9297,0.9427,0.9525
3,deep CNN Optimizer(SGD),48.22,51.13,49.77,1.2879,1.2163,1.2412
4,deep CNN Optimizer(RMSprop),51.57,52.00,49.57,1.2501,1.2267,1.2492
5,deep CNN Batchsize(16)),48.22,51.13,49.77,1.2879,1.2163,1.2412


### Batchsize(64)

In [147]:
deep_cnn = Sequential()

# Block 1
deep_cnn.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 2
deep_cnn.add(Conv2D(64, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 3
deep_cnn.add(Conv2D(128, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Block 4
deep_cnn.add(Conv2D(256, (3,3), activation="relu"))
deep_cnn.add(MaxPooling2D((2,2)))

# Fully Connected Layer
deep_cnn.add(Flatten())
deep_cnn.add(Dense(256, activation="relu"))

# Output Layer (7 diseases)
deep_cnn.add(Dense(7, activation="softmax"))

In [148]:
deep_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [114]:
train_generator_64 = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical"
)
val_generator_64 = test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)
test_generator_64 = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [150]:
history_deep_batch64 = deep_cnn.fit(
    train_generator_64,
    validation_data=val_generator_64,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 130s 1s/step - accuracy: 0.5023 - loss: 1.2952 - val_accuracy: 0.5013 - val_loss: 1.2415
Epoch 2/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 135s 1s/step - accuracy: 0.5101 - loss: 1.2540 - val_accuracy: 0.5386 - val_loss: 1.1619
Epoch 3/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 130s 1s/step - accuracy: 0.5043 - loss: 1.2396 - val_accuracy: 0.5067 - val_loss: 1.2494
Epoch 4/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 130s 1s/step - accuracy: 0.5324 - loss: 1.1924 - val_accuracy: 0.5499 - val_loss: 1.0862
Epoch 5/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - accuracy: 0.5271 - loss: 1.1969 - val_accuracy: 0.6065 - val_loss: 1.0455
Epoch 6/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 130s 1s/step - accuracy: 0.5378 - loss: 1.1736 - val_accuracy: 0.5506 - val_loss: 1.0965
Epoch 7/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 132s 1s/step - accuracy: 0.5367 - loss: 1.1497 - val_accuracy: 0.5912 - val_loss: 1.0542
Epoch 8/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - accuracy: 0.5459 - loss: 1.1422 - val_accuracy: 0.

In [151]:
rain_loss, train_acc = deep_cnn.evaluate(train_generator_64)
val_loss, val_acc = deep_cnn.evaluate(val_generator_64)
test_loss, test_acc = deep_cnn.evaluate(test_generator_64)

110/110 ━━━━━━━━━━━━━━━━━━━━ 56s 506ms/step - accuracy: 0.6190 - loss: 0.9528
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 309ms/step - accuracy: 0.6172 - loss: 0.9534
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 310ms/step - accuracy: 0.6148 - loss: 0.9553


In [152]:
deep_cnn.save("models2/deep_cnn_batch(64).keras")
print("saved batch(64)")

saved batch(64)


In [16]:
deep_results.append({
    "Model": "deep CNN + Batchsize(64))",
    "Train Accuracy": 61.90,
    "Validation Accuracy": 61.72,
    "Test Accuracy": 61.48,
    "Train Loss": 0.9528,
    "Validation Loss": 0.9534,
    "Test Loss": 0.9553
})

In [17]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,deep CNN Early Stopping,59.16,60.39,58.42,0.9645,0.9616,0.9742
2,deep CNN Learning Rate,60.76,60.59,60.08,0.9297,0.9427,0.9525
3,deep CNN Optimizer(SGD),48.22,51.13,49.77,1.2879,1.2163,1.2412
4,deep CNN Optimizer(RMSprop),51.57,52.00,49.57,1.2501,1.2267,1.2492
5,deep CNN Batchsize(16)),48.22,51.13,49.77,1.2879,1.2163,1.2412
6,deep CNN + Batchsize(64)),61.90,61.72,61.48,0.9528,0.9534,0.9553


## Hyper Parameter

In [19]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

IMG_SIZE = 224
NUM_CLASSES = 7

def build_deep_cnn(hp):

    model = Sequential()

    # Block 1
    model.add(
        Conv2D(
            filters=hp.Choice("filters1", [32, 64]),
            kernel_size=(3,3),
            activation="relu",
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    )
    model.add(MaxPooling2D((2,2)))

    # Block 2
    model.add(
        Conv2D(
            filters=hp.Choice("filters2", [64, 128]),
            kernel_size=(3,3),
            activation="relu"
        )
    )
    model.add(MaxPooling2D((2,2)))

    # Block 3
    model.add(
        Conv2D(
            filters=hp.Choice("filters3", [128, 256]),
            kernel_size=(3,3),
            activation="relu"
        )
    )
    model.add(MaxPooling2D((2,2)))

    # Block 4
    model.add(
        Conv2D(
            filters=hp.Choice("filters4", [256, 512]),
            kernel_size=(3,3),
            activation="relu"
        )
    )
    model.add(MaxPooling2D((2,2)))

    model.add(Flatten())

    model.add(
        Dense(
            units=hp.Choice("dense_units", [128, 256, 512]),
            activation="relu"
        )
    )

    model.add(
        Dense(NUM_CLASSES, activation="softmax")
    )

    optimizer = hp.Choice(
        "optimizer",
        ["adam", "sgd", "rmsprop"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [1e-2, 1e-3, 1e-4]
    )

    if optimizer == "adam":
        opt = Adam(learning_rate=learning_rate)

    elif optimizer == "sgd":
        opt = SGD(learning_rate=learning_rate, momentum=0.9)

    else:
        opt = RMSprop(learning_rate=learning_rate)

    model.compile(
        optimizer=opt,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [21]:
tuner = kt.RandomSearch(
    build_deep_cnn,
    objective="val_accuracy",
    max_trials=4,
    directory="tuner_results",
    project_name="deep_cnn"
)

tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=7
)

best_hp = tuner.get_best_hyperparameters(1)[0]

print(best_hp.values)

Trial 4 Complete [00h 31m 32s]
val_accuracy: 0.6697736382484436

Best val_accuracy So Far: 0.6724367737770081
Total elapsed time: 01h 54m 59s
{'filters1': 32, 'filters2': 64, 'filters3': 128, 'filters4': 512, 'dense_units': 128, 'optimizer': 'sgd', 'learning_rate': 0.001}


In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ============================================
# Deep CNN - Best Hyperparameter Model
# ============================================

deep_cnn_hp = Sequential(name="DeepCNN_Hyperparameter")

# Block 1
deep_cnn_hp.add(
    Conv2D(
        filters=32,
        kernel_size=(3,3),
        activation="relu",
        input_shape=(224,224,3)
    )
)
deep_cnn_hp.add(MaxPooling2D(pool_size=(2,2)))

# Block 2
deep_cnn_hp.add(
    Conv2D(
        filters=64,
        kernel_size=(3,3),
        activation="relu"
    )
)
deep_cnn_hp.add(MaxPooling2D(pool_size=(2,2)))

# Block 3
deep_cnn_hp.add(
    Conv2D(
        filters=128,
        kernel_size=(3,3),
        activation="relu"
    )
)
deep_cnn_hp.add(MaxPooling2D(pool_size=(2,2)))

# Block 4
deep_cnn_hp.add(
    Conv2D(
        filters=512,
        kernel_size=(3,3),
        activation="relu"
    )
)
deep_cnn_hp.add(MaxPooling2D(pool_size=(2,2)))

# Fully Connected Layer
deep_cnn_hp.add(Flatten())

deep_cnn_hp.add(
    Dense(
        units=128,
        activation="relu"
    )
)
deep_cnn_hp.add(Dropout(0.5))
# Output Layer
deep_cnn_hp.add(
    Dense(
        7,
        activation="softmax"
    )
)

# Compile

deep_cnn_hp.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Callbacks

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)


# Train Model

history_deep_hp = deep_cnn_hp.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, lr_scheduler]
)

# Evaluate

test_loss, test_accuracy = deep_cnn_hp.evaluate(test_generator)

print("="*50)
print("Deep CNN Hyperparameter Model")
print("="*50)
print(f"Test Accuracy : {test_accuracy*100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

# ============================================
# Save Model
# ============================================

deep_cnn_hp.save("models2/deep_cnn_hyperparameter_finalll.keras")

print("\nModel saved successfully!")

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 164s 741ms/step - accuracy: 0.6621 - loss: 1.2244 - val_accuracy: 0.6698 - val_loss: 1.1257 - learning_rate: 0.0010
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 167s 757ms/step - accuracy: 0.6695 - loss: 1.0975 - val_accuracy: 0.6698 - val_loss: 1.0186 - learning_rate: 0.0010
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 996ms/step - accuracy: 0.6695 - loss: 1.0428 - val_accuracy: 0.6698 - val_loss: 0.9856 - learning_rate: 0.0010
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 157s 715ms/step - accuracy: 0.6695 - loss: 1.0108 - val_accuracy: 0.6698 - val_loss: 0.9766 - learning_rate: 0.0010
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 166s 753ms/step - accuracy: 0.6700 - loss: 0.9990 - val_accuracy: 0.6698 - val_loss: 0.9870 - learning_rate: 0.0010
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 239s 1s/step - accuracy: 0.6692 - loss: 0.9815 - val_accuracy: 0.6698 - val_loss: 0.9613 - learning_rate: 0.0010
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 158s 715ms/step - accuracy:

In [18]:
deep_results.append({
    "Model": "deep CNN  HyperParameter)",
    "Train Accuracy": 65.23,
    "Validation Accuracy": 61.82,
    "Test Accuracy": 67.53,
    "Train Loss": 1.0234,
    "Validation Loss": 1.407,
    "Test Loss": 0.9553
})

In [19]:
comparison_deep = pd.DataFrame(deep_results)
comparison_deep

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,deep CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,deep CNN Early Stopping,59.16,60.39,58.42,0.9645,0.9616,0.9742
2,deep CNN Learning Rate,60.76,60.59,60.08,0.9297,0.9427,0.9525
3,deep CNN Optimizer(SGD),48.22,51.13,49.77,1.2879,1.2163,1.2412
4,deep CNN Optimizer(RMSprop),51.57,52.00,49.57,1.2501,1.2267,1.2492
5,deep CNN Batchsize(16)),48.22,51.13,49.77,1.2879,1.2163,1.2412
6,deep CNN + Batchsize(64)),61.90,61.72,61.48,0.9528,0.9534,0.9553
7,deep CNN HyperParameter),65.23,61.82,67.53,1.0234,1.4070,0.9553


In [20]:
comparison_deep.to_csv("dataset/model_comparison_deep_cnn.csv", index = False)
print("csv_saved")

csv_saved


## BATCH NORMALIZATION

In [21]:
batch_nor_results = []
batch_nor_results.append({
    "Model": "Batch normalization CNN",
    "Train Accuracy": 47.55,
    "Validation Accuracy": 48.20,
    "Test Accuracy": 46.97,
    "Train Loss": 1.5012,
    "Validation Loss": 1.4709,
    "Test Loss": 1.5244
})

In [22]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.2,46.97,1.5012,1.4709,1.5244


### Early stopping

In [19]:
from tensorflow.keras.models import load_model

In [20]:
deep_batch = Sequential(name="CNN_BatchNormalization")

# Block 1
deep_batch.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
deep_batch.add(BatchNormalization())
deep_batch.add(MaxPooling2D((2,2)))

# Block 2
deep_batch.add(Conv2D(64, (3,3), activation="relu"))
deep_batch.add(BatchNormalization())
deep_batch.add(MaxPooling2D((2,2)))

# Block 3
deep_batch.add(Conv2D(128, (3,3), activation="relu"))
deep_batch.add(BatchNormalization())
deep_batch.add(MaxPooling2D((2,2)))

# Block 4
deep_batch.add(Conv2D(256, (3,3), activation="relu"))
deep_batch.add(BatchNormalization())
deep_batch.add(MaxPooling2D((2,2)))

# Fully Connected Layer
deep_batch.add(Flatten())
deep_batch.add(Dense(256, activation="relu"))

# Output Layer
deep_batch.add(Dense(7, activation="softmax"))

In [21]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [22]:
history_deep_batch = deep_batch.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 200s 902ms/step - accuracy: 0.5417 - loss: 1.2312 - val_accuracy: 0.5646 - val_loss: 1.2398
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 911ms/step - accuracy: 0.5511 - loss: 1.2196 - val_accuracy: 0.4887 - val_loss: 1.3648
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 207s 939ms/step - accuracy: 0.5616 - loss: 1.1303 - val_accuracy: 0.5253 - val_loss: 1.2356
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 202s 918ms/step - accuracy: 0.5522 - loss: 1.1338 - val_accuracy: 0.5233 - val_loss: 1.3267
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 204s 929ms/step - accuracy: 0.5631 - loss: 1.1294 - val_accuracy: 0.5360 - val_loss: 1.3010
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 203s 923ms/step - accuracy: 0.5796 - loss: 1.0789 - val_accuracy: 0.5320 - val_loss: 1.2532
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 199s 905ms/step - accuracy: 0.5505 - loss: 1.0861 - val_accuracy: 0.3675 - val_loss: 1.8470


In [23]:
train_loss, train_acc = deep_batch.evaluate(train_generator)
val_loss, val_acc = deep_batch.evaluate(val_generator)
test_loss, test_acc = deep_batch.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 56s 256ms/step - accuracy: 0.4961 - loss: 1.3241
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 182ms/step - accuracy: 0.5253 - loss: 1.2356
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 182ms/step - accuracy: 0.5090 - loss: 1.2584


In [30]:
deep_batch.save("models2/batch_cnn_earlystopping.keras")
print("saved batch_es")

saved batch_es


In [23]:
batch_nor_results.append({
    "Model": "Batch CNN + Early Stopping",
    "Train Accuracy": 50.61,
    "Validation Accuracy": 52.53,
    "Test Accuracy": 50.90,
    "Train Loss": 1.3241,
    "Validation Loss": 1.2356,
    "Test Loss": 1.2584
})

In [24]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.20,46.97,1.5012,1.4709,1.5244
1,Batch CNN + Early Stopping,50.61,52.53,50.90,1.3241,1.2356,1.2584


### Learningrate Scheduler

In [33]:
batch_cnn_es = Sequential(name="CNN_BatchNormalization")

# Block 1
batch_cnn_es.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
batch_cnn_es.add(BatchNormalization())
batch_cnn_es.add(MaxPooling2D((2,2)))

# Block 2
batch_cnn_es.add(Conv2D(64, (3,3), activation="relu"))
batch_cnn_es.add(BatchNormalization())
batch_cnn_es.add(MaxPooling2D((2,2)))

# Block 3
batch_cnn_es.add(Conv2D(128, (3,3), activation="relu"))
batch_cnn_es.add(BatchNormalization())
batch_cnn_es.add(MaxPooling2D((2,2)))

# Block 4
batch_cnn_es.add(Conv2D(256, (3,3), activation="relu"))
batch_cnn_es.add(BatchNormalization())
batch_cnn_es.add(MaxPooling2D((2,2)))

# Fully Connected Layer
batch_cnn_es.add(Flatten())
batch_cnn_es.add(Dense(256, activation="relu"))

# Output Layer
batch_cnn_es.add(Dense(7, activation="softmax"))


In [34]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
batch_cnn_es.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [36]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

In [37]:
history_batch_lr = batch_cnn_es.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights,
    callbacks=[lr_scheduler]
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 200s 904ms/step - accuracy: 0.5278 - loss: 1.3522 - val_accuracy: 0.5919 - val_loss: 1.0292 - learning_rate: 1.0000e-04
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 205s 932ms/step - accuracy: 0.5304 - loss: 1.2752 - val_accuracy: 0.5752 - val_loss: 1.2594 - learning_rate: 1.0000e-04
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 209s 947ms/step - accuracy: 0.5384 - loss: 1.2598 - val_accuracy: 0.5945 - val_loss: 1.0606 - learning_rate: 1.0000e-04
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.5629 - loss: 1.1007 - val_accuracy: 0.5866 - val_loss: 1.0492 - learning_rate: 1.0000e-04
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 977ms/step - accuracy: 0.5675 - loss: 1.1287
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.5675 - loss: 1.1287 - val_accuracy: 0.5672 - val_loss: 1.3186 - learning_rate: 1.0000e-04
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 910ms/st

In [38]:
train_loss, train_acc = batch_cnn_es.evaluate(train_generator)
val_loss, val_acc = batch_cnn_es.evaluate(val_generator)
test_loss, test_acc = batch_cnn_es.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 58s 262ms/step - accuracy: 0.6011 - loss: 0.9803
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 185ms/step - accuracy: 0.5586 - loss: 1.1009
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 185ms/step - accuracy: 0.5675 - loss: 1.0691


In [40]:
batch_cnn_es.save("models2/deep_cnn_learningrate.keras")
print("saved deep_es")

saved deep_es


In [25]:
batch_nor_results.append({
    "Model": "Batch CNN Learningrate",
    "Train Accuracy": 60.11,
    "Validation Accuracy": 55.86,
    "Test Accuracy": 56.75,
    "Train Loss": 0.9803,
    "Validation Loss": 1.1009,
    "Test Loss": 1.0691
})

In [26]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.20,46.97,1.5012,1.4709,1.5244
1,Batch CNN + Early Stopping,50.61,52.53,50.90,1.3241,1.2356,1.2584
2,Batch CNN Learningrate,60.11,55.86,56.75,0.9803,1.1009,1.0691


## Optimizer Comparison

### 1.SGD

In [69]:
batch_normalization = Sequential(name="CNN_BatchNormalization")

# Block 1
batch_cnn_sgd.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
batch_cnn_sgd.add(BatchNormalization())
batch_cnn_sgd.add(MaxPooling2D((2,2)))

# Block 2
batch_cnn_sgd.add(Conv2D(64, (3,3), activation="relu"))
batch_cnn_sgd.add(BatchNormalization())
batch_cnn_sgd.add(MaxPooling2D((2,2)))

# Block 3
batch_cnn_sgd.add(Conv2D(128, (3,3), activation="relu"))
batch_cnn_sgd.add(BatchNormalization())
batch_cnn_sgd.add(MaxPooling2D((2,2)))

# Block 4
batch_cnn_sgd.add(Conv2D(256, (3,3), activation="relu"))
batch_cnn_sgd.add(BatchNormalization())
batch_cnn_sgd.add(MaxPooling2D((2,2)))

# Fully Connected Layer
batch_cnn_sgd.add(Flatten())
batch_cnn_sgd.add(Dense(256, activation="relu"))

# Output Layer
batch_cnn_sgd.add(Dense(7, activation="softmax"))


In [70]:
from tensorflow.keras.optimizers import SGD
 
batch_cnn_sgd.compile(
    optimizer=SGD(learning_rate=0.001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [71]:
history_batchnor_sgd = batch_cnn_sgd.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 202s 915ms/step - accuracy: 0.3916 - loss: 1.9311 - val_accuracy: 0.0846 - val_loss: 2.7628
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 199s 905ms/step - accuracy: 0.4044 - loss: 1.7378 - val_accuracy: 0.2883 - val_loss: 1.8641
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 196s 889ms/step - accuracy: 0.4739 - loss: 1.5527 - val_accuracy: 0.5326 - val_loss: 1.2699
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 205s 930ms/step - accuracy: 0.4642 - loss: 1.4916 - val_accuracy: 0.4720 - val_loss: 1.3946
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 198s 900ms/step - accuracy: 0.4693 - loss: 1.4906 - val_accuracy: 0.3029 - val_loss: 1.8063
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 885ms/step - accuracy: 0.4920 - loss: 1.3829 - val_accuracy: 0.5293 - val_loss: 1.2610
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 199s 904ms/step - accuracy: 0.5024 - loss: 1.3842 - val_accuracy: 0.5073 - val_loss: 1.1488
Epoch 8/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 203s 923ms/step - accuracy: 0.5195 - loss: 1

In [75]:
train_loss, train_acc = batch_cnn_sgd.evaluate(train_generator)
val_loss, val_acc = batch_cnn_sgd.evaluate(val_generator)
test_loss, test_acc = batch_cnn_sgd.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 57s 260ms/step - accuracy: 0.5294 - loss: 1.1462
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 191ms/step - accuracy: 0.5353 - loss: 1.1120
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 191ms/step - accuracy: 0.5223 - loss: 1.1724


In [119]:
batch_cnn_sgd.save("models2/batch_cnn_learningrate.keras")
print("saved deep_es")

saved deep_es


In [27]:
batch_nor_results.append({
    "Model": "Batch CNN Optimization(SGD)",
    "Train Accuracy": 52.94,
    "Validation Accuracy": 53.53,
    "Test Accuracy": 52.53,
    "Train Loss": 1.1462,
    "Validation Loss": 1.1120,
    "Test Loss": 1.1724
})

In [28]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.20,46.97,1.5012,1.4709,1.5244
1,Batch CNN + Early Stopping,50.61,52.53,50.90,1.3241,1.2356,1.2584
2,Batch CNN Learningrate,60.11,55.86,56.75,0.9803,1.1009,1.0691
3,Batch CNN Optimization(SGD),52.94,53.53,52.53,1.1462,1.1120,1.1724


### 2.RMSprop

In [79]:
batch_cnn_rmsprop = Sequential(name="CNN_BatchNormalization")

# Block 1
batch_cnn_rmsprop.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
batch_cnn_rmsprop.add(BatchNormalization())
batch_cnn_rmsprop.add(MaxPooling2D((2,2)))

# Block 2
batch_cnn_rmsprop.add(Conv2D(64, (3,3), activation="relu"))
batch_cnn_rmsprop.add(BatchNormalization())
batch_cnn_rmsprop.add(MaxPooling2D((2,2)))

# Block 3
batch_cnn_rmsprop.add(Conv2D(128, (3,3), activation="relu"))
batch_cnn_rmsprop.add(BatchNormalization())
batch_cnn_rmsprop.add(MaxPooling2D((2,2)))

# Block 4
batch_cnn_rmsprop.add(Conv2D(256, (3,3), activation="relu"))
batch_cnn_rmsprop.add(BatchNormalization())
batch_cnn_rmsprop.add(MaxPooling2D((2,2)))

# Fully Connected Layer
batch_cnn_rmsprop.add(Flatten())
batch_cnn_rmsprop.add(Dense(256, activation="relu"))

# Output Layer
batch_cnn_rmsprop.add(Dense(7, activation="softmax"))


In [80]:
from tensorflow.keras.optimizers import RMSprop
 
batch_cnn_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [81]:
history_batch_rmsprop = batch_cnn_rmsprop.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 910ms/step - accuracy: 0.2942 - loss: 9.7162 - val_accuracy: 0.2676 - val_loss: 4.1814
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 197s 895ms/step - accuracy: 0.3214 - loss: 5.5239 - val_accuracy: 0.1977 - val_loss: 2.2944
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 207s 938ms/step - accuracy: 0.3134 - loss: 3.3970 - val_accuracy: 0.2996 - val_loss: 2.0408
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 204s 924ms/step - accuracy: 0.3586 - loss: 2.1234 - val_accuracy: 0.3795 - val_loss: 1.9958
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 205s 932ms/step - accuracy: 0.3502 - loss: 2.6904 - val_accuracy: 0.1438 - val_loss: 20.8054
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 208s 946ms/step - accuracy: 0.3906 - loss: 1.9722 - val_accuracy: 0.3003 - val_loss: 1.9518
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 208s 943ms/step - accuracy: 0.3785 - loss: 1.7929 - val_accuracy: 0.6305 - val_loss: 2.7671
Epoch 8/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 202s 916ms/step - accuracy: 0.3509 - loss: 

In [83]:
train_loss, train_acc = batch_cnn_rmsprop.evaluate(train_generator)
val_loss, val_acc = batch_cnn_rmsprop.evaluate(val_generator)
test_loss, test_acc = batch_cnn_rmsprop.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 60s 274ms/step - accuracy: 0.4354 - loss: 1.4673
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 190ms/step - accuracy: 0.4421 - loss: 1.4319
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 187ms/step - accuracy: 0.4245 - loss: 1.4975


In [120]:
batch_cnn_rmsprop.save("models2/batch_cnn_learningrate.keras")
print("saved batch_")

saved batch_


In [29]:
batch_nor_results.append({
    "Model": "Batch CNN Optimization(RMSprop)",
    "Train Accuracy": 43.54,
    "Validation Accuracy": 44.21,
    "Test Accuracy": 42.45,
    "Train Loss": 1.4673,
    "Validation Loss": 1.4319,
    "Test Loss": 1.4975
})

In [30]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.20,46.97,1.5012,1.4709,1.5244
1,Batch CNN + Early Stopping,50.61,52.53,50.90,1.3241,1.2356,1.2584
2,Batch CNN Learningrate,60.11,55.86,56.75,0.9803,1.1009,1.0691
3,Batch CNN Optimization(SGD),52.94,53.53,52.53,1.1462,1.1120,1.1724
4,Batch CNN Optimization(RMSprop),43.54,44.21,42.45,1.4673,1.4319,1.4975


### BATCH SIZE COMPARISON

### Batchsize(16)

In [95]:
batch_cnn_batch16 = Sequential(name="CNN_BatchNormalization")

# Block 1
batch_cnn_batch16.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
batch_cnn_batch16.add(BatchNormalization())
batch_cnn_batch16.add(MaxPooling2D((2,2)))

# Block 2
batch_cnn_batch16.add(Conv2D(64, (3,3), activation="relu"))
batch_cnn_batch16.add(BatchNormalization())
batch_cnn_batch16.add(MaxPooling2D((2,2)))

# Block 3
batch_cnn_batch16.add(Conv2D(128, (3,3), activation="relu"))
batch_cnn_batch16.add(BatchNormalization())
batch_cnn_batch16.add(MaxPooling2D((2,2)))

# Block 4
batch_cnn_batch16.add(Conv2D(256, (3,3), activation="relu"))
batch_cnn_batch16.add(BatchNormalization())
batch_cnn_batch16.add(MaxPooling2D((2,2)))

# Fully Connected Layer
batch_cnn_batch16.add(Flatten())
batch_cnn_batch16.add(Dense(256, activation="relu"))

# Output Layer
batch_cnn_batch16.add(Dense(7, activation="softmax"))


In [96]:
from tensorflow.keras.optimizers import Adam
batch_cnn_batch16.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [97]:
history_batch_batch16 = batch_cnn_batch16.fit(
    train_generator_16,
    validation_data=val_generator_16,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 201s 454ms/step - accuracy: 0.5247 - loss: 1.4736 - val_accuracy: 0.5007 - val_loss: 1.4429
Epoch 2/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 204s 465ms/step - accuracy: 0.5193 - loss: 1.3804 - val_accuracy: 0.5286 - val_loss: 1.2353
Epoch 3/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 204s 465ms/step - accuracy: 0.5116 - loss: 1.3335 - val_accuracy: 0.4887 - val_loss: 1.3130
Epoch 4/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 220s 502ms/step - accuracy: 0.5334 - loss: 1.3233 - val_accuracy: 0.4787 - val_loss: 1.4257
Epoch 5/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 203s 462ms/step - accuracy: 0.5405 - loss: 1.2731 - val_accuracy: 0.5533 - val_loss: 1.2098
Epoch 6/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 205s 467ms/step - accuracy: 0.5408 - loss: 1.2742 - val_accuracy: 0.4134 - val_loss: 1.5629
Epoch 7/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 205s 467ms/step - accuracy: 0.5512 - loss: 1.1779 - val_accuracy: 0.5885 - val_loss: 1.0727
Epoch 8/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 203s 462ms/step - accuracy: 0.5544 - loss: 1

In [98]:
train_loss, train_acc = batch_cnn_batch16.evaluate(train_generator_16)
val_loss, val_acc = batch_cnn_batch16.evaluate(val_generator_16)
test_loss, test_acc = batch_cnn_batch16.evaluate(test_generator_16)

439/439 ━━━━━━━━━━━━━━━━━━━━ 57s 130ms/step - accuracy: 0.6168 - loss: 0.9756
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 94ms/step - accuracy: 0.6172 - loss: 1.0195
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step - accuracy: 0.6168 - loss: 1.0072


In [121]:
batch_cnn_batch16.save("models2/batch_cnn_batch16.keras")
print("saved batch16")

saved batch16


In [31]:
batch_nor_results.append({
    "Model": "batch CNN Batchsize(16))",
    "Train Accuracy": 61.68,
    "Validation Accuracy": 61.72,
    "Test Accuracy": 61.68 ,
    "Train Loss": 0.9756,
    "Validation Loss": 0.9756,
    "Test Loss": 1.0072
})

In [32]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.20,46.97,1.5012,1.4709,1.5244
1,Batch CNN + Early Stopping,50.61,52.53,50.90,1.3241,1.2356,1.2584
2,Batch CNN Learningrate,60.11,55.86,56.75,0.9803,1.1009,1.0691
3,Batch CNN Optimization(SGD),52.94,53.53,52.53,1.1462,1.1120,1.1724
4,Batch CNN Optimization(RMSprop),43.54,44.21,42.45,1.4673,1.4319,1.4975
5,batch CNN Batchsize(16)),61.68,61.72,61.68,0.9756,0.9756,1.0072


### Batchsize(64)

In [115]:
batch_cnn_batch16 = Sequential(name="CNN_BatchNormalization")

# Block 1
batch_cnn_batch64.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
batch_cnn_batch64.add(BatchNormalization())
batch_cnn_batch64.add(MaxPooling2D((2,2)))

# Block 2
batch_cnn_batch64.add(Conv2D(64, (3,3), activation="relu"))
batch_cnn_batch64.add(BatchNormalization())
batch_cnn_batch64.add(MaxPooling2D((2,2)))

# Block 3
batch_cnn_batch64.add(Conv2D(128, (3,3), activation="relu"))
batch_cnn_batch64.add(BatchNormalization())
batch_cnn_batch64.add(MaxPooling2D((2,2)))

# Block 4
batch_cnn_batch64.add(Conv2D(256, (3,3), activation="relu"))
batch_cnn_batch64.add(BatchNormalization())
batch_cnn_batch64.add(MaxPooling2D((2,2)))

# Fully Connected Layer
batch_cnn_batch64.add(Flatten())
batch_cnn_batch64.add(Dense(256, activation="relu"))

# Output Layer
batch_cnn_batch64.add(Dense(7, activation="softmax"))

In [116]:
batch_cnn_batch64.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [118]:
history_batch_batch64 = batch_cnn_batch64.fit(
    train_generator_64,
    validation_data=val_generator_64,
    epochs=8,
    class_weight=class_weights
)

Epoch 1/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 211s 2s/step - accuracy: 0.5599 - loss: 1.2356 - val_accuracy: 0.5120 - val_loss: 1.3811
Epoch 2/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 207s 2s/step - accuracy: 0.5593 - loss: 1.1428 - val_accuracy: 0.5819 - val_loss: 1.0696
Epoch 3/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 207s 2s/step - accuracy: 0.5770 - loss: 1.1029 - val_accuracy: 0.5499 - val_loss: 1.2546
Epoch 4/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step - accuracy: 0.5512 - loss: 1.1820 - val_accuracy: 0.5240 - val_loss: 1.2166
Epoch 5/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step - accuracy: 0.5914 - loss: 1.0835 - val_accuracy: 0.6258 - val_loss: 1.0109
Epoch 6/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 215s 2s/step - accuracy: 0.5736 - loss: 1.0342 - val_accuracy: 0.6245 - val_loss: 0.9594
Epoch 7/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.5880 - loss: 1.0116 - val_accuracy: 0.5253 - val_loss: 1.2236
Epoch 8/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 208s 2s/step - accuracy: 0.6143 - loss: 0.9952 - val_accuracy: 0.

In [123]:
train_loss, train_acc = batch_cnn_batch64.evaluate(train_generator_64)
val_loss, val_acc = batch_cnn_batch64.evaluate(val_generator_64)
test_loss, test_acc = batch_cnn_batch64.evaluate(test_generator_64)

110/110 ━━━━━━━━━━━━━━━━━━━━ 62s 563ms/step - accuracy: 0.4157 - loss: 1.6180
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 352ms/step - accuracy: 0.4567 - loss: 1.4393
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 354ms/step - accuracy: 0.4398 - loss: 1.4686


In [ ]:
batch_cnn_batch64.save("models2/batch_cnn_batch64.keras")
print("saved batch64")

In [33]:
batch_nor_results.append({
    "Model": "batch CNN Batchsize(64))",
    "Train Accuracy": 41.57 ,
    "Validation Accuracy": 45.67,
    "Test Accuracy": 43.98 ,
    "Train Loss": 1.6180,
    "Validation Loss": 1.4393,
    "Test Loss": 1.4686
})

In [34]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.20,46.97,1.5012,1.4709,1.5244
1,Batch CNN + Early Stopping,50.61,52.53,50.90,1.3241,1.2356,1.2584
2,Batch CNN Learningrate,60.11,55.86,56.75,0.9803,1.1009,1.0691
3,Batch CNN Optimization(SGD),52.94,53.53,52.53,1.1462,1.1120,1.1724
4,Batch CNN Optimization(RMSprop),43.54,44.21,42.45,1.4673,1.4319,1.4975
5,batch CNN Batchsize(16)),61.68,61.72,61.68,0.9756,0.9756,1.0072
6,batch CNN Batchsize(64)),41.57,45.67,43.98,1.6180,1.4393,1.4686


### HyperParameter

In [126]:
batch_cnn_hyper= load_model("models/batch_normalization.keras")

In [127]:
batch_cnn_hyper.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [128]:
history_batch_hyper = batch_cnn_hyper.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 204s 921ms/step - accuracy: 0.5696 - loss: 1.1622 - val_accuracy: 0.4414 - val_loss: 1.5445
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 206s 937ms/step - accuracy: 0.5631 - loss: 1.1204 - val_accuracy: 0.5313 - val_loss: 1.3030
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 200s 907ms/step - accuracy: 0.5615 - loss: 1.1556 - val_accuracy: 0.5606 - val_loss: 1.1582
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 206s 934ms/step - accuracy: 0.5850 - loss: 1.0505 - val_accuracy: 0.6105 - val_loss: 0.9741
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 216s 980ms/step - accuracy: 0.5940 - loss: 1.0141 - val_accuracy: 0.5712 - val_loss: 1.0502
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 210s 954ms/step - accuracy: 0.6051 - loss: 0.9779 - val_accuracy: 0.5426 - val_loss: 1.1979
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 995ms/step - accuracy: 0.6066 - loss: 0.9551 - val_accuracy: 0.5819 - val_loss: 1.1038
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 212s 961ms/step - accuracy: 0.5907 -

In [129]:
train_loss, train_acc = batch_cnn_hyper.evaluate(train_generator)
val_loss, val_acc = batch_cnn_hyper.evaluate(val_generator)
test_loss, test_acc = batch_cnn_hyper.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 58s 261ms/step - accuracy: 0.6083 - loss: 1.0164
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 182ms/step - accuracy: 0.5939 - loss: 1.0525
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 184ms/step - accuracy: 0.6041 - loss: 1.0408


In [130]:
batch_cnn_hyper.save("models2/batch_cnn_hyperparameter.keras")
print("saved batch64")

saved batch64


In [35]:
batch_nor_results.append({
    "Model": "batch CNN HyperParameter",
    "Train Accuracy": 60.83 ,
    "Validation Accuracy": 59.39,
    "Test Accuracy": 60.41 ,
    "Train Loss": 1.0164,
    "Validation Loss": 1.0525,
    "Test Loss": 1.0408
})

In [36]:
comparison_batch_nor = pd.DataFrame(batch_nor_results)
comparison_batch_nor

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Batch normalization CNN,47.55,48.20,46.97,1.5012,1.4709,1.5244
1,Batch CNN + Early Stopping,50.61,52.53,50.90,1.3241,1.2356,1.2584
2,Batch CNN Learningrate,60.11,55.86,56.75,0.9803,1.1009,1.0691
3,Batch CNN Optimization(SGD),52.94,53.53,52.53,1.1462,1.1120,1.1724
4,Batch CNN Optimization(RMSprop),43.54,44.21,42.45,1.4673,1.4319,1.4975
5,batch CNN Batchsize(16)),61.68,61.72,61.68,0.9756,0.9756,1.0072
6,batch CNN Batchsize(64)),41.57,45.67,43.98,1.6180,1.4393,1.4686
7,batch CNN HyperParameter,60.83,59.39,60.41,1.0164,1.0525,1.0408


In [37]:
comparison_batch_nor.to_csv("dataset/model_comparison_batch_cnn.csv", index = False)
print("csv_saved")

csv_saved


## ResNet50

In [57]:
resnet_comparison = []
resnet_comparison.append({
    "Model": "ResNet50",
    "Train Accuracy": 59.43,
    "Validation Accuracy": 59.25,
    "Test Accuracy": 57.55,
    "Train Loss": 1.0916,
    "Validation Loss": 1.0551,
    "Test Loss": 1.1118
})
 

In [21]:
from tensorflow.keras.applications.resnet50 import preprocess_input
# image generators
train_datagen_resnet = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,    
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8,1.2]
)
 
val_datagen_resnet = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen_resnet = ImageDataGenerator(preprocessing_function=preprocess_input)

In [22]:
# data generators
train_generator_resnet = train_datagen_resnet.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=True
)
 
val_generator_resnet = val_datagen_resnet.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)
 
test_generator_resnet = test_datagen_resnet.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [23]:
# data generators
train_generator_resnet16 = train_datagen_resnet.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=True
)
 
val_generator_resnet16 = val_datagen_resnet.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)
 
test_generator_resnet16 = test_datagen_resnet.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [24]:
import tensorflow as tf

from tensorflow.keras.applications import ResNet50

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.models import Model

from tensorflow.keras.optimizers import Adam

In [25]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
#create model
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
 
base_model.trainable = False
 
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(7, activation="softmax")(x)



In [26]:
resnet_model = Model(
    inputs=base_model.input,
    outputs=output,
    name="ResNet50"
)

In [27]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [149]:
# compile the model
resnet_model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [153]:
# train the model
history_resnet = resnet_model.fit(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=10,
    class_weight = class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.3944 - loss: 1.7717 - val_accuracy: 0.6072 - val_loss: 1.1067
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.5077 - loss: 1.4024 - val_accuracy: 0.6172 - val_loss: 1.0017
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 227s 1s/step - accuracy: 0.5448 - loss: 1.2210 - val_accuracy: 0.6105 - val_loss: 1.0395
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 235s 1s/step - accuracy: 0.5472 - loss: 1.1927 - val_accuracy: 0.6451 - val_loss: 0.9053
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 0.5509 - loss: 1.1550 - val_accuracy: 0.5959 - val_loss: 1.0471
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.5770 - loss: 1.1090 - val_accuracy: 0.5859 - val_loss: 1.1022
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 233s 1s/step - accuracy: 0.5719 - loss: 1.0812 - val_accuracy: 0.6225 - val_loss: 0.9030
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.5967 - loss: 1.0543 - val_accu

In [154]:
train_loss, train_acc = resnet_model.evaluate(train_generator_resnet)
val_loss, val_acc = resnet_model.evaluate(val_generator_resnet)
test_loss, test_acc = resnet_model.evaluate(test_generator_resnet)

220/220 ━━━━━━━━━━━━━━━━━━━━ 794s 4s/step - accuracy: 0.6864 - loss: 0.8056
47/47 ━━━━━━━━━━━━━━━━━━━━ 40s 839ms/step - accuracy: 0.6684 - loss: 0.8235
47/47 ━━━━━━━━━━━━━━━━━━━━ 88s 2s/step - accuracy: 0.6500 - loss: 0.8873


In [155]:
resnet_model.save("models2/resnet_lr_es.keras")
print("saved lr_es")

saved lr_es


In [58]:
resnet_comparison.append({
    "Model": "ResNet50 ES_LR",
    "Train Accuracy": 68.64,
    "Validation Accuracy": 66.84,
    "Test Accuracy": 65.00,
    "Train Loss": 0.8056,
    "Validation Loss": 0.8235,
    "Test Loss": 0.8873
})

In [59]:
comparison_resnet_data = pd.DataFrame(resnet_comparison)
comparison_resnet_data

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,ResNet50,59.43,59.25,57.55,1.0916,1.0551,1.1118
1,ResNet50 ES_LR,68.64,66.84,65.00,0.8056,0.8235,0.8873


## 1.SGD

In [161]:
resnet_model_sgd = Model(
    inputs=base_model.input,
    outputs=output,
    name="ResNet50"
)

In [162]:
# compile the model
resnet_model_sgd.compile(
    optimizer=SGD(learning_rate=0.001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [163]:
# train the model
history_resnet_sgd = resnet_model_sgd.fit(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=9,
    class_weight = class_weights,
    callbacks=[early_stop]
)

Epoch 1/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 225s 1s/step - accuracy: 0.6265 - loss: 0.9779 - val_accuracy: 0.6824 - val_loss: 0.7799
Epoch 2/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.5960 - loss: 0.9734 - val_accuracy: 0.6445 - val_loss: 0.8728
Epoch 3/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 243s 1s/step - accuracy: 0.6127 - loss: 0.9786 - val_accuracy: 0.6398 - val_loss: 0.8745
Epoch 4/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 1s/step - accuracy: 0.6019 - loss: 0.9876 - val_accuracy: 0.6252 - val_loss: 0.8842
Epoch 5/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 237s 1s/step - accuracy: 0.5989 - loss: 0.9782 - val_accuracy: 0.6478 - val_loss: 0.8371


In [164]:
train_loss, train_acc = resnet_model_sgd.evaluate(train_generator_resnet)
val_loss, val_acc = resnet_model_sgd.evaluate(val_generator_resnet)
test_loss, test_acc = resnet_model_sgd.evaluate(test_generator_resnet)

220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 913ms/step - accuracy: 0.7120 - loss: 0.7320
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 819ms/step - accuracy: 0.6824 - loss: 0.7799
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 821ms/step - accuracy: 0.6780 - loss: 0.8336


In [165]:
resnet_model.save("models2/resnet_SGD.keras")
print("saved sgd")

saved sgd


In [60]:
resnet_comparison.append({
    "Model": "ResNet50 SGD",
    "Train Accuracy": 71.20,
    "Validation Accuracy": 68.24 ,
    "Test Accuracy": 67.80,
    "Train Loss": 0.7320,
    "Validation Loss": 0.7799,
    "Test Loss": 0.8336
})

In [61]:
comparison_resnet_data = pd.DataFrame(resnet_comparison)
comparison_resnet_data

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,ResNet50,59.43,59.25,57.55,1.0916,1.0551,1.1118
1,ResNet50 ES_LR,68.64,66.84,65.00,0.8056,0.8235,0.8873
2,ResNet50 SGD,71.20,68.24,67.80,0.7320,0.7799,0.8336


## RMSprop

In [169]:
resnet_model_rms = Model(
    inputs=base_model.input,
    outputs=output,
    name="ResNet50"
)

In [170]:
# compile the model
resnet_model_rms.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [171]:
# train the model
history_resnet_rms = resnet_model_rms.fit(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=9,
    class_weight = class_weights,
    callbacks=[early_stop]
)

Epoch 1/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 250s 1s/step - accuracy: 0.6063 - loss: 1.0115 - val_accuracy: 0.5972 - val_loss: 0.9597
Epoch 2/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 246s 1s/step - accuracy: 0.6003 - loss: 0.9990 - val_accuracy: 0.5905 - val_loss: 0.9808
Epoch 3/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 251s 1s/step - accuracy: 0.6155 - loss: 0.9859 - val_accuracy: 0.6458 - val_loss: 0.8562
Epoch 4/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 246s 1s/step - accuracy: 0.5949 - loss: 0.9883 - val_accuracy: 0.5333 - val_loss: 1.1315
Epoch 5/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 240s 1s/step - accuracy: 0.6057 - loss: 0.9748 - val_accuracy: 0.6218 - val_loss: 0.9251
Epoch 6/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 0.6183 - loss: 0.9618 - val_accuracy: 0.6238 - val_loss: 0.8646
Epoch 7/9
220/220 ━━━━━━━━━━━━━━━━━━━━ 229s 1s/step - accuracy: 0.6000 - loss: 0.9672 - val_accuracy: 0.6438 - val_loss: 0.8805


In [172]:
train_loss, train_acc = resnet_model_rms.evaluate(train_generator_resnet)
val_loss, val_acc = resnet_model_rms.evaluate(val_generator_resnet)
test_loss, test_acc = resnet_model_rms.evaluate(test_generator_resnet)

220/220 ━━━━━━━━━━━━━━━━━━━━ 192s 872ms/step - accuracy: 0.6846 - loss: 0.7974
47/47 ━━━━━━━━━━━━━━━━━━━━ 42s 890ms/step - accuracy: 0.6458 - loss: 0.8562
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 837ms/step - accuracy: 0.6414 - loss: 0.9173


In [173]:
resnet_model_rms.save("models2/resnet_rmsprop.keras")
print("saved rmsprop")

saved rmsprop


In [62]:
resnet_comparison.append({
    "Model": "ResNet50 RMSprop",
    "Train Accuracy": 68.46,
    "Validation Accuracy":64.58,
    "Test Accuracy": 64.14,
    "Train Loss": 0.7974,
    "Validation Loss": 0.8562,
    "Test Loss": 0.9173
})

In [63]:
comparison_resnet_data = pd.DataFrame(resnet_comparison)
comparison_resnet_data

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,ResNet50,59.43,59.25,57.55,1.0916,1.0551,1.1118
1,ResNet50 ES_LR,68.64,66.84,65.00,0.8056,0.8235,0.8873
2,ResNet50 SGD,71.20,68.24,67.80,0.7320,0.7799,0.8336
3,ResNet50 RMSprop,68.46,64.58,64.14,0.7974,0.8562,0.9173


## Batchsize(16)

In [202]:
resnet_model_16 = Model(
    inputs=base_model.input,
    outputs=output,
    name="ResNet50"
)    

In [204]:
# compile the model
resnet_model_16.compile(
    optimizer=Adam(learning_rate=0.001
                  ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [206]:
# train the model
history_resnet_16 = resnet_model_16.fit(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=10,
    class_weight = class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.3845 - loss: 1.8013 - val_accuracy: 0.5240 - val_loss: 1.2968
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.4521 - loss: 1.3978 - val_accuracy: 0.6358 - val_loss: 1.0224
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 233s 1s/step - accuracy: 0.4994 - loss: 1.2885 - val_accuracy: 0.5326 - val_loss: 1.1130
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 1s/step - accuracy: 0.4906 - loss: 1.3053 - val_accuracy: 0.6112 - val_loss: 1.0165
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.5322 - loss: 1.1981 - val_accuracy: 0.6099 - val_loss: 0.9898
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 337s 2s/step - accuracy: 0.5171 - loss: 1.1776 - val_accuracy: 0.5752 - val_loss: 1.0619
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 1s/step - accuracy: 0.5331 - loss: 1.1266 - val_accuracy: 0.5799 - val_loss: 1.0309
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 226s 1s/step - accuracy: 0.5372 - loss: 1.1339 - val_accu

In [207]:
train_loss, train_acc = resnet_model_16.evaluate(train_generator_16)
val_loss, val_acc = resnet_model_16.evaluate(val_generator_16)
test_loss, test_acc = resnet_model_16.evaluate(test_generator_16)

439/439 ━━━━━━━━━━━━━━━━━━━━ 183s 416ms/step - accuracy: 0.6695 - loss: 1.2781
94/94 ━━━━━━━━━━━━━━━━━━━━ 38s 405ms/step - accuracy: 0.6698 - loss: 1.2762
94/94 ━━━━━━━━━━━━━━━━━━━━ 38s 407ms/step - accuracy: 0.6693 - loss: 1.2783


In [208]:
resnet_model_16.save("models2/resnet_16.keras")
print("saved 16")

saved 16


In [64]:
resnet_comparison.append({
    "Model": "ResNet50 batchsize(16)",
    "Train Accuracy": 66.95,
    "Validation Accuracy": 66.98,
    "Test Accuracy": 66.93,
    "Train Loss": 1.2781,
    "Validation Loss":1.2762 ,
    "Test Loss": 1.2783
})

In [65]:
comparison_resnet_data = pd.DataFrame(resnet_comparison)
comparison_resnet_data

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,ResNet50,59.43,59.25,57.55,1.0916,1.0551,1.1118
1,ResNet50 ES_LR,68.64,66.84,65.00,0.8056,0.8235,0.8873
2,ResNet50 SGD,71.20,68.24,67.80,0.7320,0.7799,0.8336
3,ResNet50 RMSprop,68.46,64.58,64.14,0.7974,0.8562,0.9173
4,ResNet50 batchsize(16),66.95,66.98,66.93,1.2781,1.2762,1.2783


## Hyperparameter

In [28]:
def build_resnet_model(hp):

    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(
        hp.Choice("dense_units",[128,256,512]),
        activation="relu"
    )(x)
    x = Dropout(
        hp.Choice("dropout",[0.3,0.5])
    )(x)
    output = Dense(
        7,
        activation="softmax"
    )(x)
    resnet_model = Model(
        inputs=base_model.input,
        outputs=output
    )
    lr = hp.Choice(
        "learning_rate",
        [1e-3,1e-4,1e-5]
    )
    optimizer_name = hp.Choice(
        "optimizer",
        ["adam","sgd","rmsprop"]
    )
    if optimizer_name=="adam":
        optimizer=Adam(lr)
    elif optimizer_name=="sgd":
        optimizer=SGD(
            lr,
            momentum=0.9
        )
    else:
        optimizer=RMSprop(lr)

    resnet_model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return resnet_model

In [29]:
!pip3 install keras-tuner

In [30]:
import keras_tuner as kt
tuner = kt.RandomSearch(
    build_resnet_model, 
    objective="val_accuracy",
    max_trials=5, 
    directory="tuner",
    project_name="basic_cnn"
 
)

In [34]:
pip install tensorboard

  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 32.3 MB/s  0:00:00 13.2 MB/s eta 0:00:01
Using cached tensorboard_data_server-0.7.2-py3-none-any.whl (2.4 kB)
Using cached markdown-3.10.2-py3-none-any.whl (108 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [tensorboard]━━━━━━ 2/3 [tensorboard]
Note: you may need to restart the kernel to use updated packages.


In [36]:
import keras_tuner as kt
from tensorflow.keras.optimizers import RMSprop
tuner.search(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=10
)

Trial 5 Complete [00h 00m 01s]

Best val_accuracy So Far: 0.7922769784927368
Total elapsed time: 01h 17m 17s


In [37]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

{'dense_units': 128, 'dropout': 0.3, 'learning_rate': 0.001, 'optimizer': 'rmsprop'}


In [38]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Base Model
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# Classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(7, activation="softmax")(x)

resnet_final = Model(
    inputs=base_model.input,
    outputs=output
)

# Compile using best hyperparameters
resnet_final.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [39]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7
)

In [40]:
history_resnet_final = resnet_final.fit(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 229s 1s/step - accuracy: 0.4827 - loss: 1.6942 - val_accuracy: 0.5646 - val_loss: 1.0687 - learning_rate: 0.0010
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 235s 1s/step - accuracy: 0.5536 - loss: 1.3740 - val_accuracy: 0.6039 - val_loss: 0.9618 - learning_rate: 0.0010
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - accuracy: 0.5795 - loss: 1.2898 - val_accuracy: 0.6378 - val_loss: 0.9555 - learning_rate: 0.0010
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 236s 1s/step - accuracy: 0.5983 - loss: 1.2549 - val_accuracy: 0.5812 - val_loss: 1.0635 - learning_rate: 0.0010
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 236s 1s/step - accuracy: 0.6093 - loss: 1.1661 - val_accuracy: 0.6458 - val_loss: 0.9654 - learning_rate: 0.0010
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 233s 1s/step - accuracy: 0.6437 - loss: 1.0434 - val_accuracy: 0.6565 - val_loss: 0.8749 - learning_rate: 2.0000e-04
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 233s 1s/step - accuracy: 0.6411 - loss

In [42]:
train_loss, train_acc = resnet_final.evaluate(train_generator_resnet)
val_loss, val_acc = resnet_final.evaluate(val_generator_resnet)
test_loss, test_acc = resnet_final.evaluate(test_generator_resnet)

220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 829ms/step - accuracy: 0.7090 - loss: 0.7208
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 810ms/step - accuracy: 0.6798 - loss: 0.8130
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 811ms/step - accuracy: 0.6600 - loss: 0.8929


In [44]:
resnet_final.save("models2/resnet_final_model.keras")
print("saved final")

saved final


In [66]:
resnet_comparison.append({
    "Model": "ResNet50 HyperParameter",
    "Train Accuracy": 70.90,
    "Validation Accuracy": 67.98,
    "Test Accuracy": 66.00,
    "Train Loss": 0.7208,
    "Validation Loss":0.8130 ,
    "Test Loss": 0.8929
})

In [67]:
comparison_resnet_data = pd.DataFrame(resnet_comparison)
comparison_resnet_data

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,ResNet50,59.43,59.25,57.55,1.0916,1.0551,1.1118
1,ResNet50 ES_LR,68.64,66.84,65.00,0.8056,0.8235,0.8873
2,ResNet50 SGD,71.20,68.24,67.80,0.7320,0.7799,0.8336
3,ResNet50 RMSprop,68.46,64.58,64.14,0.7974,0.8562,0.9173
4,ResNet50 batchsize(16),66.95,66.98,66.93,1.2781,1.2762,1.2783
5,ResNet50 HyperParameter,70.90,67.98,66.00,0.7208,0.8130,0.8929


## FineTuning

In [212]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
resnet_finetune = Model(
        inputs=base_model.input,
        outputs=output
    )
# Unfreeze the model
resnet_finetune.trainable = True

# Freeze all layers except the last 30
for layer in resnet_finetune.layers[:-30]:
    layer.trainable = False
    
# Compile
resnet_finetune.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Callbacks
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)
# Train
history_resnet_ft = resnet_finetune.fit(
    train_generator_resnet,
    validation_data=val_generator_resnet,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.5789 - loss: 1.1103 - val_accuracy: 0.6398 - val_loss: 0.9267 - learning_rate: 1.0000e-05
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 232s 1s/step - accuracy: 0.5987 - loss: 1.0925 - val_accuracy: 0.6385 - val_loss: 0.9366 - learning_rate: 1.0000e-05
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 878ms/step - accuracy: 0.6011 - loss: 1.1050
Epoch 3: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
220/220 ━━━━━━━━━━━━━━━━━━━━ 235s 1s/step - accuracy: 0.6011 - loss: 1.1050 - val_accuracy: 0.6385 - val_loss: 0.9423 - learning_rate: 1.0000e-05
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.5944 - loss: 1.0986 - val_accuracy: 0.6365 - val_loss: 0.9480 - learning_rate: 2.0000e-06
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 834ms/step - accuracy: 0.5944 - loss: 1.0747
Epoch 5: ReduceLROnPlateau reducing learning rate to 3.999999989900971e-07.
220/220 ━━━━━━━━━━━━━━━━━━━━ 222s 1s/step - accuracy: 

In [213]:
train_loss, train_acc = resnet_finetune.evaluate(train_generator_resnet)
val_loss, val_acc = resnet_finetune.evaluate(val_generator_resnet)
test_loss, test_acc = resnet_finetune.evaluate(test_generator_resnet)

220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 876ms/step - accuracy: 0.6525 - loss: 0.9471
47/47 ━━━━━━━━━━━━━━━━━━━━ 40s 838ms/step - accuracy: 0.6398 - loss: 0.9267
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 818ms/step - accuracy: 0.6234 - loss: 0.9849


In [214]:
resnet_model_16.save("models2/resnet_hyper.keras")
print("saved hyper")

saved hyper


In [68]:
resnet_comparison.append({
    "Model": "ResNet50 tuning",
    "Train Accuracy": 65.25,
    "Validation Accuracy": 63.38,
    "Test Accuracy": 62.34,
    "Train Loss": 0.9471,
    "Validation Loss":0.9267,
    "Test Loss": 0.9849
})

In [69]:
comparison_resnet_data = pd.DataFrame(resnet_comparison)
comparison_resnet_data

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,ResNet50,59.43,59.25,57.55,1.0916,1.0551,1.1118
1,ResNet50 ES_LR,68.64,66.84,65.00,0.8056,0.8235,0.8873
2,ResNet50 SGD,71.20,68.24,67.80,0.7320,0.7799,0.8336
3,ResNet50 RMSprop,68.46,64.58,64.14,0.7974,0.8562,0.9173
4,ResNet50 batchsize(16),66.95,66.98,66.93,1.2781,1.2762,1.2783
5,ResNet50 HyperParameter,70.90,67.98,66.00,0.7208,0.8130,0.8929
6,ResNet50 tuning,65.25,63.38,62.34,0.9471,0.9267,0.9849


In [70]:
comparison_resnet_data.to_csv("dataset/model_comparison_ResNet50.csv", index = False)
print("csv_saved")

csv_saved


## DenseNetV2

In [113]:
DenseNet121_data=[]
DenseNet121_data.append({
    "Model": "DenseNet121",
    "Train Accuracy": 48.73,
    "Validation Accuracy": 58.72,
    "Test Accuracy": 56.65,
    "Train Loss": 1.3829,
    "Validation Loss": 1.2098,
    "Test Loss": 1.2557
})

In [71]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input


# image generators
train_datagen_dense = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8,1.2]
)

val_datagen_dense = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen_dense = ImageDataGenerator(preprocessing_function=preprocess_input)


In [72]:
# data generators
train_generator_dense = train_datagen_dense.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=True
)

val_generator_dense = val_datagen_dense.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

test_generator_dense = test_datagen_dense.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [73]:
from tensorflow.keras.layers import GlobalAveragePooling2D

# model
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(7, activation="softmax")(x)


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [74]:
from tensorflow.keras.callbacks import EarlyStopping
# callback
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [75]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

# create the scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=2,
    min_lr=1e-6, verbose=1
)

In [76]:
densenet_model_es = Model(
    inputs=base_model.input,
    outputs=output,
    name="DenseNet121"
)


In [77]:
densenet_model_es.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [78]:
history_densenet_model_es = densenet_model_es.fit(
    train_generator_dense,
    validation_data=val_generator_dense,
    epochs=10,
    class_weight=class_weights,
    callbacks=[lr_scheduler, early_stop],
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 996ms/step - accuracy: 0.2066 - loss: 2.1875 - val_accuracy: 0.3156 - val_loss: 1.7051 - learning_rate: 1.0000e-04
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 216s 983ms/step - accuracy: 0.3474 - loss: 1.7689 - val_accuracy: 0.4501 - val_loss: 1.4769 - learning_rate: 1.0000e-04
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 216s 983ms/step - accuracy: 0.4295 - loss: 1.5834 - val_accuracy: 0.5939 - val_loss: 1.2720 - learning_rate: 1.0000e-04
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 986ms/step - accuracy: 0.4571 - loss: 1.5025 - val_accuracy: 0.5799 - val_loss: 1.2738 - learning_rate: 1.0000e-04
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 953ms/step - accuracy: 0.4949 - loss: 1.4248
Epoch 5: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
220/220 ━━━━━━━━━━━━━━━━━━━━ 247s 1s/step - accuracy: 0.4949 - loss: 1.4248 - val_accuracy: 0.5240 - val_loss: 1.3169 - learning_rate: 1.0000e-04
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 218s 989m

In [79]:
train_loss, train_accuracy = densenet_model_es.evaluate(train_generator_dense)
val_loss, val_accuracy = densenet_model_es.evaluate(val_generator_dense)
test_loss, test_accuracy = densenet_model_es.evaluate(test_generator_dense)

220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 829ms/step - accuracy: 0.5767 - loss: 1.2366
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 797ms/step - accuracy: 0.5919 - loss: 1.1927
47/47 ━━━━━━━━━━━━━━━━━━━━ 37s 795ms/step - accuracy: 0.5662 - loss: 1.2348


In [81]:
densenet_model_es.save("models2/densenet_model_es.keras")
print("model saved")

model saved


In [114]:
DenseNet121_data.append({
    "Model": "DenseNet121 ES_LR",
    "Train Accuracy": 57.67 ,
    "Validation Accuracy": 59.19,
    "Test Accuracy": 56.62,
    "Train Loss": 1.2366,
    "Validation Loss": 1.1927,
    "Test Loss": 1.2348

})

## SGD

In [86]:
densenet_model_SGD = Model(
    inputs=base_model.input,
    outputs=output,
    name="DenseNet121"
)


In [88]:
from tensorflow.keras.optimizers import SGD
densenet_model_SGD.compile(
    optimizer=SGD( learning_rate=0.01, momentum=0.9, nesterov=True),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [90]:
history_densenet_model_SGD = densenet_model_SGD.fit(
    train_generator_dense,
    validation_data=val_generator_dense,
    epochs=7,
    class_weight=class_weights,
    callbacks=[lr_scheduler, early_stop],
)

Epoch 1/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 220s 999ms/step - accuracy: 0.2227 - loss: 1.9785 - val_accuracy: 0.3469 - val_loss: 1.8380 - learning_rate: 0.0100
Epoch 2/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 216s 982ms/step - accuracy: 0.1870 - loss: 1.8931 - val_accuracy: 0.5313 - val_loss: 1.6559 - learning_rate: 0.0100
Epoch 3/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 216s 984ms/step - accuracy: 0.1927 - loss: 1.9432 - val_accuracy: 0.4467 - val_loss: 1.5846 - learning_rate: 0.0100
Epoch 4/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 984ms/step - accuracy: 0.1141 - loss: 1.9275 - val_accuracy: 0.0899 - val_loss: 2.0146 - learning_rate: 0.0100
Epoch 5/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 815ms/step - accuracy: 0.1706 - loss: 1.9298
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0019999999552965165.
220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 985ms/step - accuracy: 0.1706 - loss: 1.9298 - val_accuracy: 0.0306 - val_loss: 2.0275 - learning_rate: 0.0100
Epoch 6/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 996ms/step - accuracy: 0.122

In [91]:
train_loss, train_accuracy = densenet_model_SGD.evaluate(train_generator_dense)
val_loss, val_accuracy = densenet_model_SGD.evaluate(val_generator_dense)
test_loss, test_accuracy = densenet_model_SGD.evaluate(test_generator_dense)

220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 814ms/step - accuracy: 0.4146 - loss: 1.6536
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 828ms/step - accuracy: 0.4467 - loss: 1.5846
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 815ms/step - accuracy: 0.4385 - loss: 1.6072


In [92]:
densenet_model_SGD.save("models2/densenet_model_SGD.keras")
print("saved")

saved


In [115]:
DenseNet121_data.append({
    "Model": "DenseNet121 SGD",
    "Train Accuracy": 41.46,
    "Validation Accuracy": 44.67,
    "Test Accuracy": 43.85,
    "Train Loss": 1.5846,
    "Validation Loss": 1.6536,
    "Test Loss": 1.6072
})

## RMSprop

In [94]:
densenet_model_RMSprop = Model(
    inputs=base_model.input,
    outputs=output,
    name="DenseNet121"
)


In [95]:

densenet_model_RMSprop.compile(
    optimizer=SGD(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [96]:
history_densenet_model_RMSprop = densenet_model_RMSprop.fit(
    train_generator_dense,
    validation_data=val_generator_dense,
    epochs=10,
    class_weight=class_weights,
    callbacks=[lr_scheduler, early_stop],
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 999ms/step - accuracy: 0.2285 - loss: 1.8967 - val_accuracy: 0.4474 - val_loss: 1.5842 - learning_rate: 1.0000e-04
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 215s 979ms/step - accuracy: 0.2322 - loss: 1.9063 - val_accuracy: 0.4474 - val_loss: 1.5865 - learning_rate: 1.0000e-04
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 809ms/step - accuracy: 0.2367 - loss: 1.8945
Epoch 3: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
220/220 ━━━━━━━━━━━━━━━━━━━━ 243s 1s/step - accuracy: 0.2367 - loss: 1.8945 - val_accuracy: 0.4481 - val_loss: 1.5854 - learning_rate: 1.0000e-04
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1093s 5s/step - accuracy: 0.2312 - loss: 1.8981 - val_accuracy: 0.4487 - val_loss: 1.5854 - learning_rate: 2.0000e-05
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [97]:
train_loss, train_accuracy = densenet_model_RMSprop.evaluate(train_generator_dense)
val_loss, val_accuracy = densenet_model_RMSprop.evaluate(val_generator_dense)
test_loss, test_accuracy = densenet_model_RMSprop.evaluate(test_generator_dense)

220/220 ━━━━━━━━━━━━━━━━━━━━ 213s 968ms/step - accuracy: 0.4144 - loss: 1.6594
47/47 ━━━━━━━━━━━━━━━━━━━━ 43s 908ms/step - accuracy: 0.4474 - loss: 1.5842
47/47 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.4385 - loss: 1.6068


In [98]:
densenet_model_RMSprop.save("models2/densenet_model_RMSprop.keras")
print("saved")

saved


In [116]:
DenseNet121_data.append({
    "Model": "DenseNet121 RMSprop",
    "Train Accuracy": 41.44,
    "Validation Accuracy": 44.74,
    "Test Accuracy": 43.85,
    "Train Loss": 1.6594,
    "Validation Loss": 1.5842,
    "Test Loss": 1.6068
})

## Batchsize(64)

In [102]:
train_generator_dense_64 = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=True
)

val_generator_dense_64 = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

test_generator_dense_64 = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [103]:
densenet_64 = Model(
    inputs=base_model.input,
    outputs=output,
    name="EfficientNetB0"
)
densenet_64.summary()

Model: "EfficientNetB0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [104]:
densenet_64.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [105]:
history_densenet_64 = densenet_64.fit(
    train_generator_dense_64,
    validation_data=val_generator_dense_64,
    epochs=6,
    class_weight=class_weights,
    callbacks=[lr_scheduler],
)

Epoch 1/6
110/110 ━━━━━━━━━━━━━━━━━━━━ 273s 2s/step - accuracy: 0.3004 - loss: 1.9400 - val_accuracy: 0.5047 - val_loss: 1.5521 - learning_rate: 1.0000e-04
Epoch 2/6
110/110 ━━━━━━━━━━━━━━━━━━━━ 228s 2s/step - accuracy: 0.3101 - loss: 1.8966 - val_accuracy: 0.4987 - val_loss: 1.5152 - learning_rate: 1.0000e-04
Epoch 3/6
110/110 ━━━━━━━━━━━━━━━━━━━━ 233s 2s/step - accuracy: 0.3639 - loss: 1.8081 - val_accuracy: 0.5379 - val_loss: 1.4087 - learning_rate: 1.0000e-04
Epoch 4/6
110/110 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.4134 - loss: 1.7459 - val_accuracy: 0.5632 - val_loss: 1.3584 - learning_rate: 1.0000e-04
Epoch 5/6
110/110 ━━━━━━━━━━━━━━━━━━━━ 222s 2s/step - accuracy: 0.4368 - loss: 1.6678 - val_accuracy: 0.5373 - val_loss: 1.3590 - learning_rate: 1.0000e-04
Epoch 6/6
110/110 ━━━━━━━━━━━━━━━━━━━━ 231s 2s/step - accuracy: 0.4388 - loss: 1.6132 - val_accuracy: 0.5506 - val_loss: 1.3179 - learning_rate: 1.0000e-04


In [106]:
train_loss, train_accuracy = densenet_64.evaluate(train_generator_dense_64)
val_loss, val_accuracy = densenet_64.evaluate(val_generator_dense_64)
test_loss, test_accuracy = densenet_64.evaluate(test_generator_dense_64)

110/110 ━━━━━━━━━━━━━━━━━━━━ 184s 2s/step - accuracy: 0.5424 - loss: 1.3738
24/24 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.5506 - loss: 1.3179
24/24 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.5283 - loss: 1.3543


In [107]:
densenet_64.save("models2/densenet_model_64.keras")
print("saved")

saved


In [117]:
DenseNet121_data.append({
    "Model": "DenseNet121 batchsize(64)",
    "Train Accuracy": 54.24,
    "Validation Accuracy": 55.06,
    "Test Accuracy": 52.83,
    "Train Loss": 1.3738,
    "Validation Loss": 1.3179,
    "Test Loss":1.3543
})

## FineTuning

In [110]:
from tensorflow.keras.applications import DenseNet121
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(7, activation="softmax")(x)

densenet_ft = Model(
    inputs=base_model.input,
    outputs=output
)

densenet_ft.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

# Train
history_densenet_ft = densenet_ft.fit(
    train_generator_dense,
    validation_data=val_generator_dense,
    epochs=8,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - accuracy: 0.1601 - loss: 2.4059 - val_accuracy: 0.1798 - val_loss: 2.0182 - learning_rate: 1.0000e-05
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 1s/step - accuracy: 0.1854 - loss: 2.1548 - val_accuracy: 0.2716 - val_loss: 1.8647 - learning_rate: 1.0000e-05
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 225s 1s/step - accuracy: 0.2243 - loss: 2.0165 - val_accuracy: 0.3349 - val_loss: 1.7758 - learning_rate: 1.0000e-05
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.2573 - loss: 1.9245 - val_accuracy: 0.3708 - val_loss: 1.7069 - learning_rate: 1.0000e-05
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - accuracy: 0.2803 - loss: 1.8185 - val_accuracy: 0.3828 - val_loss: 1.6687 - learning_rate: 1.0000e-05
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 232s 1s/step - accuracy: 0.3024 - loss: 1.7883 - val_accuracy: 0.4481 - val_loss: 1.5678 - learning_rate: 1.0000e-05
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 

In [111]:
train_loss, train_accuracy = densenet_ft.evaluate(train_generator_dense)
val_loss, val_accuracy = densenet_ft.evaluate(val_generator_dense)
test_loss, test_accuracy = densenet_ft.evaluate(test_generator_dense)

220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 856ms/step - accuracy: 0.5077 - loss: 1.5058
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 829ms/step - accuracy: 0.4900 - loss: 1.4751
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 811ms/step - accuracy: 0.4943 - loss: 1.4842


In [112]:
densenet_ft.save("models2/densenet_finetuned.keras")
print("densenet Fine-Tuned model saved successfully.")

densenet Fine-Tuned model saved successfully.


In [118]:
DenseNet121_data.append({
    "Model": "DenseNet121 Finetuning",
    "Train Accuracy": 59.63,
    "Validation Accuracy": 53.40,
    "Test Accuracy": 50.77,
    "Train Loss": 1.2568,
    "Validation Loss": 1.2456,
    "Test Loss": 1.2901
})

## HyperParameter

In [121]:
import keras_tuner as kt

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

def build_DenseNet121(hp):

    base_model = DenseNet121(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Fine-tune last layers
    base_model.trainable = True

    fine_tune = hp.Choice(
        "fine_tune_layers",
        values=[20, 30, 40]
    )

    for layer in base_model.layers[:-fine_tune]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    x = Dense(
        units=hp.Choice(
            "dense_units",
            values=[128, 256, 512]
        ),
        activation="relu"
    )(x)

    x = Dropout( hp.Choice("dropout", values=[0.3, 0.4, 0.5 ]) )(x)
    outputs = Dense(7, activation="softmax")(x)

    DenseNet121_hyper = Model(
        inputs=base_model.input,
        outputs=outputs
    )

    optimizer = hp.Choice(
        "optimizer",
        values=["adam", "sgd", "rmsprop"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        values=[1e-3, 1e-4, 1e-5]
    )

    if optimizer == "adam":
        opt = Adam(learning_rate=learning_rate)

    elif optimizer == "sgd":
        opt = SGD(
            learning_rate=learning_rate,
            momentum=0.9
        )

    else:
        opt = RMSprop(
            learning_rate=learning_rate
        )

    DenseNet121_hyper.compile(
        optimizer=opt,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return DenseNet121_hyper

In [122]:
tuner = kt.RandomSearch(
    build_DenseNet121,
    objective="val_accuracy",
    max_trials=3,
    directory="efficientnet_tuner",
    project_name="efficientnet_hp"
)
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7
)

In [123]:
tuner.search(
    train_generator_dense,
    validation_data=val_generator_dense,
    epochs=8,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

Trial 3 Complete [00h 42m 05s]
val_accuracy: 0.5938748121261597

Best val_accuracy So Far: 0.6937416791915894
Total elapsed time: 01h 42m 13s


In [126]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

{'fine_tune_layers': 20, 'dense_units': 128, 'dropout': 0.3, 'optimizer': 'adam', 'learning_rate': 0.0001}


In [127]:
best_model = tuner.get_best_models(1)[0]
best_model.save("models2/densenet121_hyperparameter.keras")

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [129]:
import tensorflow as tf

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)


# 20 layers
base_model.trainable = True
fine_tune_layers = 10

for layer in base_model.layers[:-fine_tune_layers]:
    layer.trainable = False


# Add classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(
    128,
    activation="relu"
)(x)
x = Dropout(0.3)(x)
output = Dense(
    7,
    activation="softmax"
)(x)

# Final Model
DenseNet121_best = Model(
    inputs=base_model.input,
    outputs=output
)

DenseNet121_best.compile(
    optimizer=Adam(
        learning_rate=0.0001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


DenseNet121_best.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_4    │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d_4… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_5    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_5… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,169,607 (27.35 MB)

 Trainable params: 300,231 (1.15 MB)

 Non-trainable params: 6,869,376 (26.20 MB)

In [130]:
history_densenet = DenseNet121_best.fit(
    train_generator_dense,
    validation_data=val_generator_dense,
    epochs=8,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        reduce_lr
    ]
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.3120 - loss: 1.9562 - val_accuracy: 0.5266 - val_loss: 1.4252 - learning_rate: 1.0000e-04
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 220s 999ms/step - accuracy: 0.4716 - loss: 1.4934 - val_accuracy: 0.5786 - val_loss: 1.2337 - learning_rate: 1.0000e-04
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 218s 991ms/step - accuracy: 0.5140 - loss: 1.3444 - val_accuracy: 0.6518 - val_loss: 1.0386 - learning_rate: 1.0000e-04
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 226s 1s/step - accuracy: 0.5703 - loss: 1.2135 - val_accuracy: 0.6565 - val_loss: 0.9851 - learning_rate: 1.0000e-04
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 238s 1s/step - accuracy: 0.5976 - loss: 1.1354 - val_accuracy: 0.6285 - val_loss: 1.0285 - learning_rate: 1.0000e-04
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 0.6150 - loss: 1.0345 - val_accuracy: 0.6352 - val_loss: 1.0167 - learning_rate: 1.0000e-04


In [131]:
DenseNet121_best.save("DenseNet121_best_model.keras")
print('saved')

saved


In [132]:
train_loss, train_acc = DenseNet121_best.evaluate(train_generator_dense,verbose=1)
val_loss, val_acc = DenseNet121_best.evaluate( val_generator_dense,verbose=1)
test_loss, test_acc = DenseNet121_best.evaluate(test_generator_dense,verbose=1)


220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 822ms/step - accuracy: 0.6268 - loss: 1.0652
47/47 ━━━━━━━━━━━━━━━━━━━━ 42s 879ms/step - accuracy: 0.6565 - loss: 0.9851
47/47 ━━━━━━━━━━━━━━━━━━━━ 42s 892ms/step - accuracy: 0.6500 - loss: 1.0090


In [133]:
DenseNet121_data.append({
    "Model": "DenseNet121 HyperParameter",
    "Train Accuracy": 62.58,
    "Validation Accuracy": 65.65,
    "Test Accuracy": 65.00 ,
    "Train Loss": 1.0652,
    "Validation Loss": 0.9851,
    "Test Loss": 1.0090
})

In [134]:
comparison_DenseNet121_datas = pd.DataFrame(DenseNet121_data)
comparison_DenseNet121_datas

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,DenseNet121,48.73,58.72,56.65,1.3829,1.2098,1.2557
1,DenseNet121 ES_LR,57.67,59.19,56.62,1.2366,1.1927,1.2348
2,DenseNet121 SGD,41.46,44.67,43.85,1.5846,1.6536,1.6072
3,DenseNet121 RMSprop,41.44,44.74,43.85,1.6594,1.5842,1.6068
4,DenseNet121 batchsize(64),54.24,55.06,52.83,1.3738,1.3179,1.3543
5,DenseNet121 Finetuning,59.63,53.40,50.77,1.2568,1.2456,1.2901
6,DenseNet121 HyperParameter,62.58,65.65,65.00,1.0652,0.9851,1.0090


In [135]:
comparison_DenseNet121_datas.to_csv("dataset/model_comparison_DenseNet121.csv", index = False)
print("csv_saved")

csv_saved


# EfficientNet (Hyper Parameter)

In [32]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [33]:
# create new data generator
train_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [34]:
train_generator_eff = train_datagen_eff.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=True
)

val_generator_eff = val_datagen_eff.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

test_generator_eff = test_datagen_eff.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [35]:

# Base Model
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

# Fine Tuning
base_model.trainable = True

# Freeze all layers except last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

# Classification Head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(7, activation="softmax")(x)

efficientnet_final_hyper = Model(
    inputs=base_model.input,
    outputs=output
)

# Compile with best optimizer and learning rate
efficientnet_final_hyper.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Callbacks
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

# Train Final Model
history_efficientnet_hp_final = efficientnet_final_hyper.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 106s 461ms/step - accuracy: 0.2476 - loss: 1.7677 - val_accuracy: 0.4134 - val_loss: 1.5741 - learning_rate: 0.0010
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 99s 449ms/step - accuracy: 0.4407 - loss: 1.3778 - val_accuracy: 0.5280 - val_loss: 1.2466 - learning_rate: 0.0010
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 98s 446ms/step - accuracy: 0.5330 - loss: 1.1357 - val_accuracy: 0.6012 - val_loss: 1.0390 - learning_rate: 0.0010
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 100s 456ms/step - accuracy: 0.5820 - loss: 1.0325 - val_accuracy: 0.6252 - val_loss: 0.9416 - learning_rate: 0.0010
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 101s 456ms/step - accuracy: 0.6100 - loss: 0.9479 - val_accuracy: 0.6372 - val_loss: 0.8766 - learning_rate: 0.0010
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 142s 647ms/step - accuracy: 0.6438 - loss: 0.8771 - val_accuracy: 0.6711 - val_loss: 0.8412 - learning_rate: 0.0010
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 522ms/step - accuracy

In [36]:
train_loss, train_acc = efficientnet_final_hyper.evaluate(train_generator_eff,verbose=1)
val_loss, val_acc = efficientnet_final_hyper.evaluate( val_generator_eff,verbose=1)
test_loss, test_acc = efficientnet_final_hyper.evaluate(test_generator_eff,verbose=1)


220/220 ━━━━━━━━━━━━━━━━━━━━ 69s 311ms/step - accuracy: 0.7148 - loss: 0.7116
47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 302ms/step - accuracy: 0.6644 - loss: 0.7996
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 531ms/step - accuracy: 0.6627 - loss: 0.8552


In [37]:
efficientnet_final_hyper.save("effiecientbo_best_model.keras")
print('saved')

saved
